# Lab Day 1 — Xây dựng mạng nơ-ron và thí nghiệm huấn luyện
**Track 4 · Ngày 1 · VinUniversity AICB 2026**

Notebook hoàn chỉnh thực hiện từ Part 0 đến Part 4, hỗ trợ chạy từng ô hoặc *Restart & Run All*.

In [1]:
# ===== 1. Cấu hình môi trường và đường dẫn =====
import os, sys, json, time, subprocess
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

# Định vị thư mục gốc repo
if Path('scripts/split_data.py').exists():
    REPO_ROOT = '.'
    OUT_DIR = '.'
elif Path('../scripts/split_data.py').exists():
    REPO_ROOT = '..'
    OUT_DIR = '..'
else:
    REPO_ROOT = '../..'
    OUT_DIR = '..'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

os.makedirs(f'{OUT_DIR}/figures', exist_ok=True)
os.makedirs(f'{OUT_DIR}/results', exist_ok=True)

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

PyTorch version: 2.12.0+cu132
Using device: cuda
GPU: NVIDIA GeForce GTX 1650


## Part 0 — Dữ liệu
- Đảm bảo đã chạy `scripts/split_data.py` (tạo `data/processed/train.npz` và `eval.npz`).
- Tách validation 20% từ train (phân tầng theo nhãn, seed 42 cố định).
- Chuẩn hoá 10 cột liên tục chỉ bằng mean/std của phần train còn lại.

In [2]:
# Nạp và chuẩn bị dữ liệu lên device
processed_path = f'{REPO_ROOT}/data/processed'
data = prepare_data(str(device), val_fraction=0.2, seed=42, processed_dir=processed_path)

print('\n--- Kiểm tra tensor shape ---')
print(f'X_tr: {data["X_tr"].shape}, y_tr: {data["y_tr"].shape}')
print(f'X_val: {data["X_val"].shape}, y_val: {data["y_val"].shape}')
print(f'X_eval: {data["X_eval"].shape}, y_eval: {data["y_eval"].shape}')

# Kiểm tra chuẩn hoá
mean_10 = data['X_tr'][:, :10].mean(dim=0).cpu().numpy()
std_10 = data['X_tr'][:, :10].std(dim=0).cpu().numpy()
print(f'Mean 10 cột số train: min={mean_10.min():.4f}, max={mean_10.max():.4f}')
print(f'Std  10 cột số train: min={std_10.min():.4f}, max={std_10.max():.4f}')
print(f'Đoán lớp đa số trên val: accuracy = {data["majority_acc"]:.4f}')

Data ready on cuda: Train=371,847, Val=92,962, Eval=116,203
Chiến lược đoán lớp đa số (lớp 1) trên val: accuracy = 0.4876

--- Kiểm tra tensor shape ---
X_tr: torch.Size([371847, 54]), y_tr: torch.Size([371847])
X_val: torch.Size([92962, 54]), y_val: torch.Size([92962])
X_eval: torch.Size([116203, 54]), y_eval: torch.Size([116203])
Mean 10 cột số train: min=-0.0003, max=0.0001
Std  10 cột số train: min=0.9999, max=1.0001
Đoán lớp đa số trên val: accuracy = 0.4876


## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Kiểm tra 5 phép thử: shape, số tham số (47 879), loss bước 0, quá khớp 20 mẫu, gradient flow.

In [3]:
# 1. Model M-base & số tham số
m_base = MLP(hidden=(256, 128), dropout=0.0, init='he').to(device)
p_cnt = count_params(m_base)
print(f'M-base: {p_cnt:,} tham số (kỳ vọng: 47,879)')
assert p_cnt == EXPECTED_PARAMS[(256, 128)]

# 2. Forward batch (8, 54)
x_dummy = torch.randn(8, 54, device=device)
logits_dummy = m_base(x_dummy)
print(f'Input (8, 54) -> Logits: {logits_dummy.shape}')
assert logits_dummy.shape == (8, 7)

# 3. Loss bước 0 trên val
step0_res = evaluate(m_base, data['X_val'], data['y_val'], loss_name='ce')
print(f'Loss bước 0: {step0_res["loss"]:.4f} (lý thuyết ln 7 ≈ {np.log(7):.4f})')

# 4. Quá khớp 20 mẫu
x_20 = data['X_tr'][:20]
y_20 = data['y_tr'][:20]
m_fit20 = MLP(hidden=(256, 128), dropout=0.0, init='he').to(device)
opt_20 = torch.optim.Adam(m_fit20.parameters(), lr=0.01)
for _ in range(250):
    opt_20.zero_grad()
    l20 = torch.nn.functional.cross_entropy(m_fit20(x_20), y_20)
    l20.backward()
    opt_20.step()
print(f'Quá khớp 20 mẫu: loss cuối = {l20.item():.6f}')
assert l20.item() < 0.01

# 5. Gradient flow
m_flow = MLP(hidden=(256, 128), dropout=0.0, init='he').to(device)
loss_flow = torch.nn.functional.cross_entropy(m_flow(x_dummy), torch.zeros(8, dtype=torch.int64, device=device))
loss_flow.backward()
all_grad_ok = True
for name, p in m_flow.named_parameters():
    if p.grad is None or p.grad.norm().item() == 0:
        all_grad_ok = False
print(f'Tất cả tham số đều có gradient chảy: {all_grad_ok}')
assert all_grad_ok

M-base: 47,879 tham số (kỳ vọng: 47,879)
Input (8, 54) -> Logits: torch.Size([8, 7])
Loss bước 0: 2.1905 (lý thuyết ln 7 ≈ 1.9459)
Quá khớp 20 mẫu: loss cuối = 0.000004
Tất cả tham số đều có gradient chảy: True


**Nhận xét Part 1**: Model đúng cấu trúc quy định (47 879 params), loss bước 0 ≈ 1.946 khớp ln 7, quá khớp 20 mẫu thành công và gradient chảy tới mọi lớp.

## Part 2 — Pipeline và Baseline (Đo độ nhiễu 3 seed)
Baseline: M-base, He init, CE loss, SGD+momentum 0.9, lr=0.05, batch 512, 20 epochs.
Chạy 3 seed để đo độ lệch chuẩn và ngưỡng 2σ.

In [4]:
base_seeds = [1, 2, 3]
base_results = []
all_results = []

for s in base_seeds:
    cfg_base = {
        **DEFAULT_CFG,
        'exp_id': f'base-s{s}',
        'group': 'baseline',
        'description': f'Baseline M-base (Seed {s})',
        'lr': 0.05,
        'seed': s,
        'epochs': 20,
    }
    print(f'\n--- Chạy {cfg_base["exp_id"]} ---')
    res = run_experiment(cfg_base, data)
    save_result(res, f'{OUT_DIR}/results')
    plot_run(res, f'{OUT_DIR}/figures/{cfg_base["exp_id"]}.png')
    base_results.append(res)
    all_results.append(res)
    print(f'Val Acc: {res["summary"]["val_acc"]:.4f}, Val Macro-F1: {res["summary"]["val_macro_f1"]:.4f}')

f1_list = [r['summary']['val_macro_f1'] for r in base_results]
acc_list = [r['summary']['val_acc'] for r in base_results]
mean_f1 = float(np.mean(f1_list))
std_f1 = float(np.std(f1_list))
two_sigma = 2 * std_f1

print('\n' + '='*40)
print(f'Baseline Macro-F1: {mean_f1:.4f} ± {std_f1:.4f}')
print(f'Baseline Accuracy: {np.mean(acc_list):.4f} ± {np.std(acc_list):.4f}')
print(f'Ngưỡng nhiễu 2σ: {two_sigma:.4f}')
print('='*40)


--- Chạy base-s1 ---
Val Acc: 0.8992, Val Macro-F1: 0.8382

--- Chạy base-s2 ---
Val Acc: 0.8975, Val Macro-F1: 0.8285

--- Chạy base-s3 ---
Val Acc: 0.9035, Val Macro-F1: 0.8434

Baseline Macro-F1: 0.8367 ± 0.0062
Baseline Accuracy: 0.9001 ± 0.0025
Ngưỡng nhiễu 2σ: 0.0124


## Part 3 — Thí nghiệm các chủ đề (Mỗi lần chỉ đổi 1 biến số)
Menu 7 chủ đề: Loss, Optimizer, Kiến trúc (Hparam), Dropout, Clipping, Mixed Precision (AMP), Init.

In [5]:
experiment_list = [
    # 1. Loss
    {'exp_id': 'loss-mse', 'group': 'loss', 'description': 'MSE Loss', 'loss': 'mse'},
    
    # 2. Optimizer
    {'exp_id': 'opt-sgd', 'group': 'optimizer', 'description': 'SGD no momentum', 'optimizer': 'sgd', 'lr': 0.05},
    {'exp_id': 'opt-adam', 'group': 'optimizer', 'description': 'Adam lr=1e-3', 'optimizer': 'adam', 'lr': 0.001},
    {'exp_id': 'opt-adamw', 'group': 'optimizer', 'description': 'AdamW wd=0.01', 'optimizer': 'adamw', 'lr': 0.001, 'weight_decay': 0.01},
    
    # 3. Hyper-parameters: Kiến trúc
    {'exp_id': 'arch-m-wide', 'group': 'hparam', 'description': 'M-wide (512-256)', 'hidden': (512, 256)},
    {'exp_id': 'arch-m-deep', 'group': 'hparam', 'description': 'M-deep (256-128-64)', 'hidden': (256, 128, 64)},
    
    # 4. Dropout
    {'exp_id': 'drop-0.1', 'group': 'dropout', 'description': 'Dropout q=0.1', 'dropout': 0.1},
    {'exp_id': 'drop-0.3', 'group': 'dropout', 'description': 'Dropout q=0.3', 'dropout': 0.3},
    
    # 5. Gradient Clipping
    {'exp_id': 'clip-1.0-normlr', 'group': 'clipping', 'description': 'Clip norm 1.0 (lr 0.05)', 'clip_norm': 1.0},
    {'exp_id': 'clip-none-highlr', 'group': 'clipping', 'description': 'No clip high lr 0.5', 'lr': 0.5, 'clip_norm': None},
    {'exp_id': 'clip-1.0-highlr', 'group': 'clipping', 'description': 'Clip 1.0 high lr 0.5', 'lr': 0.5, 'clip_norm': 1.0},
    
    # 6. Mixed precision (AMP)
    {'exp_id': 'amp-fp16', 'group': 'amp', 'description': 'Mixed Precision FP16', 'precision': 'fp16'},
    
    # 7. Khởi tạo tham số
    {'exp_id': 'init-zeros', 'group': 'init', 'description': 'Zeros Init', 'init': 'zeros'},
    {'exp_id': 'init-normal', 'group': 'init', 'description': 'Normal Init std=0.01', 'init': 'normal'},
    {'exp_id': 'init-xavier', 'group': 'init', 'description': 'Xavier Normal Init', 'init': 'xavier'},
]

for exp_spec in experiment_list:
    cfg = {**DEFAULT_CFG, 'lr': 0.05, 'epochs': 20, 'seed': 1, **exp_spec}
    print(f'\n>>> Thực thi [{cfg["exp_id"]}]: {cfg["description"]}')
    res = run_experiment(cfg, data)
    save_result(res, f'{OUT_DIR}/results')
    plot_run(res, f'{OUT_DIR}/figures/{cfg["exp_id"]}.png')
    all_results.append(res)
    
    delta = res['summary']['val_macro_f1'] - mean_f1
    sig_str = 'VƯỢT NHIỄU' if abs(delta) > two_sigma else 'nằm trong nhiễu'
    print(f'Val Macro-F1: {res["summary"]["val_macro_f1"]:.4f} (Δ={delta:+.4f}, {sig_str})')


>>> Thực thi [loss-mse]: MSE Loss
Val Macro-F1: 0.6901 (Δ=-0.1466, VƯỢT NHIỄU)

>>> Thực thi [opt-sgd]: SGD no momentum
Val Macro-F1: 0.7111 (Δ=-0.1256, VƯỢT NHIỄU)

>>> Thực thi [opt-adam]: Adam lr=1e-3
Val Macro-F1: 0.8472 (Δ=+0.0105, nằm trong nhiễu)

>>> Thực thi [opt-adamw]: AdamW wd=0.01
Val Macro-F1: 0.8446 (Δ=+0.0079, nằm trong nhiễu)

>>> Thực thi [arch-m-wide]: M-wide (512-256)
Val Macro-F1: 0.8563 (Δ=+0.0196, VƯỢT NHIỄU)

>>> Thực thi [arch-m-deep]: M-deep (256-128-64)
Val Macro-F1: 0.8553 (Δ=+0.0186, VƯỢT NHIỄU)

>>> Thực thi [drop-0.1]: Dropout q=0.1
Val Macro-F1: 0.8203 (Δ=-0.0164, VƯỢT NHIỄU)

>>> Thực thi [drop-0.3]: Dropout q=0.3
Val Macro-F1: 0.7500 (Δ=-0.0867, VƯỢT NHIỄU)

>>> Thực thi [clip-1.0-normlr]: Clip norm 1.0 (lr 0.05)
Val Macro-F1: 0.8263 (Δ=-0.0104, nằm trong nhiễu)

>>> Thực thi [clip-none-highlr]: No clip high lr 0.5
Val Macro-F1: 0.8227 (Δ=-0.0140, VƯỢT NHIỄU)

>>> Thực thi [clip-1.0-highlr]: Clip 1.0 high lr 0.5
Val Macro-F1: 0.8394 (Δ=+0.0027, nằm tr

In [6]:
# Vẽ biểu đồ so sánh theo nhóm
groups = sorted(list(set(r['cfg'].get('group') for r in all_results if r['cfg'].get('group') != 'baseline')))
for g in groups:
    group_res = [r for r in all_results if r['cfg'].get('group') == g or r['cfg'].get('exp_id') == 'base-s1']
    if len(group_res) > 1:
        plot_compare(group_res, 'val_loss', f'{OUT_DIR}/figures/compare_{g}.png', title=f'So sánh Val Loss - Nhóm {g.upper()}')
        print(f'Đã tạo ảnh so sánh: {OUT_DIR}/figures/compare_{g}.png')

Đã tạo ảnh so sánh: ../figures/compare_amp.png
Đã tạo ảnh so sánh: ../figures/compare_clipping.png
Đã tạo ảnh so sánh: ../figures/compare_dropout.png
Đã tạo ảnh so sánh: ../figures/compare_hparam.png
Đã tạo ảnh so sánh: ../figures/compare_init.png
Đã tạo ảnh so sánh: ../figures/compare_loss.png
Đã tạo ảnh so sánh: ../figures/compare_optimizer.png


## Part 4 — Đánh giá cuối trên tập Eval, Bảng Excel & Nộp bài
- Chọn cấu hình tốt nhất hoàn toàn dựa trên validation.
- Nạp best state, dự đoán trên toàn bộ `eval` -> `predictions_eval.csv`.
- Chạy `scripts/evaluate.py` -> `eval_result.json`.

In [7]:
# 1. Chọn cấu hình tốt nhất theo val_macro_f1
best_exp = max(all_results, key=lambda r: r['summary']['val_macro_f1'])
print(f'Mô hình tốt nhất: {best_exp["cfg"]["exp_id"]} (Val Macro-F1: {best_exp["summary"]["val_macro_f1"]:.4f})')

# 2. Dự đoán tập eval
pred_path = f'{OUT_DIR}/predictions_eval.csv'
final_eval(best_exp['cfg'], best_exp, data, pred_path)

# 3. Đánh giá bằng scripts/evaluate.py
eval_json_path = f'{OUT_DIR}/eval_result.json'
cmd = [
    sys.executable, f'{REPO_ROOT}/scripts/evaluate.py',
    '--pred', pred_path,
    '--out', eval_json_path,
    '--data', f'{REPO_ROOT}/data/covtype.csv.gz',
    '--meta', f'{REPO_ROOT}/data/split_metadata.csv'
]
res_cmd = subprocess.run(cmd, capture_output=True, text=True)
print(res_cmd.stdout)
if res_cmd.returncode != 0:
    print('LỖI EVAL:', res_cmd.stderr)

with open(eval_json_path, 'r', encoding='utf-8') as f:
    eval_result = json.load(f)
print(f'KẾT QUẢ EVAL: Macro-F1 = {eval_result["macro_f1"]:.4f}, Accuracy = {eval_result["accuracy"]:.4f}')

Mô hình tốt nhất: arch-m-wide (Val Macro-F1: 0.8563)
Đã ghi dự đoán eval vào ../predictions_eval.csv (116,203 dòng)

LỖI EVAL: Traceback (most recent call last):
  File "d:\create\vin\K4-Track4-Day1-Neuralnetwork\scripts\evaluate.py", line 121, in <module>
    main()
    ~~~~^^
  File "d:\create\vin\K4-Track4-Day1-Neuralnetwork\scripts\evaluate.py", line 67, in main
    truth = load_eval_labels(args.data, args.meta)
  File "d:\create\vin\K4-Track4-Day1-Neuralnetwork\scripts\evaluate.py", line 37, in load_eval_labels
    meta = pd.read_csv(meta_path)
  File "c:\Users\Admin\anaconda3\envs\DL\Lib\site-packages\pandas\io\parsers\readers.py", line 873, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "c:\Users\Admin\anaconda3\envs\DL\Lib\site-packages\pandas\io\parsers\readers.py", line 300, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "c:\Users\Admin\anaconda3\envs\DL\Lib\site-packages\pandas\io\parsers\readers.py", line 1645, in __init__
    self._e

FileNotFoundError: [Errno 2] No such file or directory: '../eval_result.json'

In [ ]:
# 4. Phân tích chi tiết lỗi trên tập Eval theo từng lớp
import pandas as pd
classes = [f'Lớp {i}' for i in range(7)]
df_perf = pd.DataFrame({
    'Lớp': classes,
    'Support': [p['support'] for p in eval_result['per_class']],
    'Precision': [round(p['precision'], 4) for p in eval_result['per_class']],
    'Recall': [round(p['recall'], 4) for p in eval_result['per_class']],
    'F1-score': [round(p['f1'], 4) for p in eval_result['per_class']],
})
print('\n--- Hiệu năng theo từng lớp trên tập Eval ---')
print(df_perf.to_string(index=False))

print('\n--- Ma trận nhầm lẫn (Confusion Matrix) ---')
df_cm = pd.DataFrame(eval_result['confusion_matrix'], index=classes, columns=classes)
print(df_cm)


In [ ]:
# 5. Điền toàn bộ kết quả vào experiments.xlsx
loaded_runs = load_results(f'{OUT_DIR}/results')
table_rows = []
for r in loaded_runs:
    eval_sc = eval_result if r['cfg']['exp_id'] == best_exp['cfg']['exp_id'] else None
    table_rows.append(to_row(r, eval_scores=eval_sc))

template_file = f'{REPO_ROOT}/templates/experiment_table_template.xlsx'
out_excel = f'{OUT_DIR}/experiments.xlsx'
write_xlsx(table_rows, template_file, out_excel)
print(f'Đã ghi thành công {len(table_rows)} dòng vào {out_excel}!')